# LAB-4: Múltiplos Robôs com Coordenação

Neste laboratório, implementaremos:
- **Sistema multi-robô** (4 robôs independentes)
- **Evitação entre robôs**
- **Alvo compartilhado ou individual**
- **Comunicação entre robôs** (distâncias relativas)

In [ ]:
# CÓDIGO LAB-4: Múltiplos Robôs com Coordenação
import pygame
import math
import numpy as np
from collections import deque

LARGURA, ALTURA = 1000, 750
FPS = 60
COR_FUNDO = (20, 24, 30)
COR_OBSTACULO = (180, 50, 50)
COR_RAIO_LIVRE = (0, 255, 100)
COR_RAIO_COLISAO = (255, 200, 0)
COR_ALVO = (255, 100, 100)
CORES_ROBO = [(0, 200, 255), (255, 0, 255), (0, 255, 255), (255, 255, 0)]

class MultiRobot:
    """Robô que faz parte de um sistema multi-robô."""
    
    def __init__(self, id, x, y, theta=0.0, cor=(0, 200, 255)):
        self.id = id
        self.x = float(x)
        self.y = float(y)
        self.theta = float(theta)
        self.cor = cor
        
        # Sensores
        self.num_sensores = 8
        self.sensor_angles = [2 * math.pi * i / self.num_sensores for i in range(self.num_sensores)]
        self.sensor_range = 150.0
        self.sensor_readings = [self.sensor_range] * self.num_sensores
        
        # Navegação
        self.target_x = None
        self.target_y = None
        self.velocidade = 0.0
        self.velocidade_max = 2.5
        self.aceleracao = 0.08
        
        # Sensores de outros robôs
        self.outros_robos = []
        self.distancia_outros = []
        self.distancia_seguranca_robo = 40  # Distância mínima entre robôs
        
        # Trajetória
        self.trajetoria = deque(maxlen=300)

    def cast_rays(self, obstacles):
        """Verifica interseção dos raios."""
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            dx = math.cos(angle)
            dy = math.sin(angle)
            
            min_distance = self.sensor_range
            
            for obs in obstacles:
                distance = self._ray_rect_intersection(self.x, self.y, dx, dy, obs)
                if distance is not None and distance < min_distance:
                    min_distance = distance
            
            self.sensor_readings.append(min_distance)

    def _ray_rect_intersection(self, x0, y0, dx, dy, rect):
        """Calcula interseção ray-rectangle."""
        rx, ry, rw, rh = rect
        x_min, x_max = rx, rx + rw
        y_min, y_max = ry, ry + rh
        
        t_min = float('-inf')
        t_max = float('inf')
        
        if abs(dx) > 1e-6:
            t1 = (x_min - x0) / dx
            t2 = (x_max - x0) / dx
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif x0 < x_min or x0 > x_max:
            return None
        
        if abs(dy) > 1e-6:
            t1 = (y_min - y0) / dy
            t2 = (y_max - y0) / dy
            if t1 > t2:
                t1, t2 = t2, t1
            t_min = max(t_min, t1)
            t_max = min(t_max, t2)
        elif y0 < y_min or y0 > y_max:
            return None
        
        if t_min < t_max and t_min > 0 and t_min < self.sensor_range:
            return t_min
        
        return None

    def set_target(self, x, y):
        """Define alvo."""
        self.target_x = x
        self.target_y = y

    def detect_other_robots(self, outros_robos):
        """Detecta posição de outros robôs."""
        self.distancia_outros = []
        for outro in outros_robos:
            if outro.id != self.id:
                dx = outro.x - self.x
                dy = outro.y - self.y
                distancia = math.sqrt(dx**2 + dy**2)
                self.distancia_outros.append((distancia, dx, dy, outro.id))

    def compute_control(self, obstacles):
        """Calcula velocidades para evitar obstáculos e outros robôs."""
        min_sensor = min(self.sensor_readings)
        
        repulsion_x = 0
        repulsion_y = 0
        distancia_min = 50
        
        # Repulsão de obstáculos
        for i, distance in enumerate(self.sensor_readings):
            if distance < distancia_min:
                angle = self.theta + self.sensor_angles[i]
                forca = (distancia_min - distance) / distancia_min
                repulsion_x -= forca * math.cos(angle)
                repulsion_y -= forca * math.sin(angle)
        
        # Repulsão de outros robôs
        for dist, dx, dy, id_outro in self.distancia_outros:
            if dist < self.distancia_seguranca_robo:
                forca = (self.distancia_seguranca_robo - dist) / self.distancia_seguranca_robo
                repulsion_x -= forca * (dx / dist)
                repulsion_y -= forca * (dy / dist)
        
        # Atração do alvo
        if self.target_x is not None:
            dx_alvo = self.target_x - self.x
            dy_alvo = self.target_y - self.y
            dist_alvo = math.sqrt(dx_alvo**2 + dy_alvo**2)
            
            if dist_alvo < 15:
                self.velocidade = 0
                return 0, 0
            
            if dist_alvo > 0:
                dx_alvo /= dist_alvo
                dy_alvo /= dist_alvo
            
            direcao_x = dx_alvo + repulsion_x * 0.03
            direcao_y = dy_alvo + repulsion_y * 0.03
            
            angle_desejado = math.atan2(direcao_y, direcao_x)
            delta_theta = angle_desejado - self.theta
            
            while delta_theta > math.pi:
                delta_theta -= 2 * math.pi
            while delta_theta < -math.pi:
                delta_theta += 2 * math.pi
            
            velocidade_angular = 0.05 * delta_theta
            
            if min_sensor < 50:
                self.velocidade = max(0, self.velocidade - self.aceleracao)
            else:
                self.velocidade = min(self.velocidade_max, self.velocidade + self.aceleracao)
            
            return self.velocidade, velocidade_angular
        
        return 0, 0

    def update(self, obstacles, outros_robos):
        """Atualiza posição."""
        self.cast_rays(obstacles)
        self.detect_other_robots(outros_robos)
        
        velocidade, velocidade_angular = self.compute_control(obstacles)
        
        self.x += velocidade * math.cos(self.theta)
        self.y += velocidade * math.sin(self.theta)
        self.theta += velocidade_angular
        
        self.x = max(0, min(LARGURA, self.x))
        self.y = max(0, min(ALTURA, self.y))
        
        self.trajetoria.append((self.x, self.y))

    def draw(self, screen):
        """Desenha o robô."""
        # Trajetória
        if len(self.trajetoria) > 1:
            cor_trajetoria = tuple(int(c * 0.4) for c in self.cor)
            pygame.draw.lines(screen, cor_trajetoria, list(self.trajetoria), 1)
        
        # Corpo
        pygame.draw.circle(screen, self.cor, (int(self.x), int(self.y)), 8)
        pygame.draw.line(screen, self.cor, (self.x, self.y),
                         (self.x + 12 * math.cos(self.theta),
                          self.y + 12 * math.sin(self.theta)), 2)
        
        # Raios (simplificados)
        for i in [0, 2, 4, 6]:
            beta = self.sensor_angles[i]
            angle = self.theta + beta
            distance = self.sensor_readings[i]
            end_x = self.x + distance * math.cos(angle)
            end_y = self.y + distance * math.sin(angle)
            
            cor = COR_RAIO_COLISAO if distance < self.sensor_range - 0.1 else COR_RAIO_LIVRE
            pygame.draw.line(screen, cor, (self.x, self.y), (end_x, end_y), 1)


def draw_obstacles(screen, obstacles):
    """Desenha obstáculos."""
    for obs in obstacles:
        x, y, w, h = obs
        pygame.draw.rect(screen, COR_OBSTACULO, (x, y, w, h))


def draw_alvo(screen, x, y, raio=12):
    """Desenha o alvo."""
    if x is not None and y is not None:
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio)
        pygame.draw.circle(screen, COR_ALVO, (int(x), int(y)), raio - 3, 2)


def main():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB-4: Múltiplos Robôs com Coordenação")
    clock = pygame.time.Clock()
    
    obstacles = [
        (100, 100, 150, 30),
        (400, 150, 30, 200),
        (650, 400, 150, 30),
        (300, 450, 250, 30),
        (50, 350, 30, 150),
        (700, 100, 50, 250),
    ]
    
    # Cria 4 robôs
    robos = [
        MultiRobot(0, 100, 100, 0, CORES_ROBO[0]),
        MultiRobot(1, 150, 150, math.pi/2, CORES_ROBO[1]),
        MultiRobot(2, 120, 120, math.pi, CORES_ROBO[2]),
        MultiRobot(3, 170, 130, 3*math.pi/2, CORES_ROBO[3]),
    ]
    
    # Define alvo compartilhado
    alvo_x, alvo_y = 850, 600
    for robo in robos:
        robo.set_target(alvo_x, alvo_y)
    
    running = True
    while running:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    running = False
            elif event.type == pygame.MOUSEBUTTONDOWN:
                x, y = pygame.mouse.get_pos()
                alvo_x, alvo_y = x, y
                for robo in robos:
                    robo.set_target(alvo_x, alvo_y)
        
        # Atualiza todos os robôs
        for robo in robos:
            robo.update(obstacles, robos)
        
        # Renderiza
        screen.fill(COR_FUNDO)
        draw_obstacles(screen, obstacles)
        draw_alvo(screen, alvo_x, alvo_y)
        
        for robo in robos:
            robo.draw(screen)
        
        # HUD
        font = pygame.font.Font(None, 18)
        for i, robo in enumerate(robos):
            text = font.render(f"Robô {robo.id}: ({robo.x:.0f}, {robo.y:.0f}) V={robo.velocidade:.1f}", 
                              True, robo.cor)
            screen.blit(text, (10, 10 + i * 22))
        
        inst_font = pygame.font.Font(None, 16)
        inst = inst_font.render("Click do mouse: Define novo alvo compartilhado | ESC: Sair", True, (150, 150, 150))
        screen.blit(inst, (10, ALTURA - 25))
        
        pygame.display.flip()
    
    pygame.quit()

if __name__ == "__main__":
    main()